Fonte:

https://www.sec.gov/dera/data/financial-statement-data-sets.html

In [ ]:
import sys
import os
sys.path
os.listdir()
os.chdir(os.getcwd().replace("\\","/").replace("/notebooks",""))
sys.path.append("src")

In [ ]:
from src.envConfig import EnvConfig
EnvConfig()

In [ ]:
DOLAR = 5.1625

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import LongType, DecimalType

#Inicializar a sessão do Spark
spark = (
    SparkSession.builder 
    .appName("Analise_SEC_BDRs")
    .getOrCreate()
)

# Usar sep="\t" porque os arquivos da SEC são separados por tabulação
df_sub = (
    spark.read.csv("asserts/sub2025.txt", header=True, sep="\t")
    .select("adsh", "cik", "name", "form", "period", "fy", "fp")
    .filter(F.col("form").isin(["10-K", "10-Q"])) # Filtra apenas balanços anuais e trimestrais
    )  

df_num = (
    spark.read.csv("asserts/num2025.txt", header=True, sep="\t") 
    .select("adsh", "tag", "version", "ddate", "value")
    )

tags_financeiras = [
    "NetIncomeLoss", 
    "StockholdersEquity", 
    "StockholdersEquityIncludingPortionAttributableToNoncontrollingInterest",
    "WeightedAverageNumberOfSharesOutstandingDiluted",
    "WeightedAverageNumberOfSharesOutstandingBasic"
]

df_num_filtered = df_num.filter(F.col("tag").isin(tags_financeiras))

df_consolidado = df_sub.join(df_num_filtered, on="adsh", how="inner")

df_pivot = (
    df_consolidado
    .groupBy("adsh", "cik", "name", "form", "period", "fy", "fp", "ddate")
    .pivot("tag")
    .agg(F.first("value").cast("double"))
)

In [ ]:
df_pivot.show(truncate=False)

In [ ]:
df_tratado = ( 
            df_pivot
            .withColumn(
                "Patrimonio_Bruto", 
                F.coalesce("StockholdersEquity", "StockholdersEquityIncludingPortionAttributableToNoncontrollingInterest")
            )
            .withColumn(
                "ações_globais",
                F.when(F.col("WeightedAverageNumberOfSharesOutstandingDiluted").isNotNull(),F.col("WeightedAverageNumberOfSharesOutstandingDiluted"))
                .otherwise(F.col("WeightedAverageNumberOfSharesOutstandingBasic"))
            )
    )

df_limpo = (
        df_tratado
        .filter(F.col("ddate") == F.col("period"))
        .withColumn("Lucro_Real", F.col("NetIncomeLoss").cast(LongType()))
        .withColumn("Patrimonio_Real", F.abs(F.col("Patrimonio_Bruto")).cast(LongType())) 
        .withColumn("lucro_brl",(F.col("Lucro_Real") * DOLAR).cast(LongType()))
        .withColumn("patrimonio_brl",(F.col("Patrimonio_Real") * DOLAR).cast(LongType()))
        .withColumn("ações_globais", F.col("ações_globais").cast(LongType()))
        .withColumn("LPA", (F.col("Lucro_Real")/F.col("ações_globais")))
        .withColumn("VPA", (F.col("Patrimonio_Real")/F.col("ações_globais")))
        .select(
                "adsh", 
                "cik", 
                "name", 
                "form", 
                "period", 
                "fy", 
                "fp", 
                "ações_globais", 
                "Lucro_Real", 
                "Patrimonio_Real", 
                "lucro_brl", 
                "patrimonio_brl",
                "LPA",
                "VPA"
            )
    )

In [ ]:
df_limpo = (
    df_limpo
    .withColumn(
        "Graham",
        F.sqrt(22.5 * (F.round(F.col("VPA"),2)) * (F.round(F.col("LPA"),2))).cast(DecimalType(38, 2))
    )
)

In [ ]:
df_limpo.filter(F.col("cik") == "320193").show(truncate=False)

In [ ]:
import pandas as pd

In [ ]:
listaBDR = pd.read_excel("asserts/BDRs Listados B3_20.07.xlsx")

In [ ]:
listaBDR["Empresa"] = listaBDR["Empresa"].str.upper()

In [ ]:
df = (
    SparkSession
    .builder
    .appName("ExcelComPandasSpark4")
    .config("spark.sql.ansi.enabled", "false")
    .config("spark.api.python.worker.connection.timeout", "200")
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")
    .getOrCreate()
)

In [ ]:
bdr = df.createDataFrame(listaBDR)

In [ ]:
df_join_bdr = bdr.join(df_limpo, on=(bdr["Empresa"] == df_limpo["name"]), how="inner")

In [ ]:
relatorio = df_join_bdr.select(
    F.col("Empresa"),
    F.col("Ticker BDR"),
    F.col("Paridade ação: BDR"),
    F.col("Setor"),
    F.col("ações_globais"),
    F.col("Lucro_Real"),
    F.col("Patrimonio_Real"),
    F.col("lucro_brl"),
    F.col("patrimonio_brl"),
    F.col("LPA"),
    F.col("VPA"),
    F.col("Graham")
)
relatorio.show(truncate=False)

In [ ]:
relatorio.filter(F.col("Ticker BDR") == "JNJB34").show(truncate=False)